# Evaluate AutoDDG technical metadata

Same spatial and temporal scoring as `03-TvdP-validate-metadata.ipynb`, applied to the technical metadata extracted from AutoDDG descriptions in `outputs/technical_from_descriptions`.

Single LLM outputs in `metadata_agent/outputs` are scored with the same rules. Their coverage is stored under `metadata`. A skipped file is a missing prediction. Every iNaturalist Single LLM file was skipped because the species CSV exceeded the context window.

ReAct outputs are the `react_pangaea_*` and `react_inaturalist_*` folders in the same directory, excluding the earlier PANGAEA smoke runs. Coverage is also under `metadata`.

iNaturalist and PANGAEA are scored separately, with the same decision rules as notebook 03:

- **iNaturalist.** Ground truth is the min/max longitude, latitude, and `eventDate` of each species CSV. A field counts as correct only when the prediction equals that value. With 100 species, the match count is the accuracy percent.
- **PANGAEA.** Ground truth is the dataset `geometryextent` and `minDateTime` / `maxDateTime` (the same fields `pangaeapy` exposes as `geometryextent`, `mintimeextent`, and `maxtimeextent`). True-positive accuracy counts exact coordinate matches (`numpy.isclose`) and same-calendar-day time matches, only where both the annotation and the prediction are present. Precision and recall treat a filled field as a positive.

A coverage object that is JSON `null` is a missing prediction. A species or dataset with no extraction file is reported and does not count as a match.

In [1]:
import json
import os
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path('/home/com3dian/Github/process_data_metadata_agent')
TECH_ROOT = REPO / 'outputs' / 'technical_from_descriptions'
INAT_DATA = REPO / 'outputs' / 'inaturalist' / 'inaturalist_100_species_1000_obs'
PANGAEA_ANNOT = REPO / 'outputs' / 'pangaea' / 'sample_20260629_163144_filtered_top5_per_topic.csv'
PANGAEA_EXTENT_CACHE = REPO / 'outputs' / 'pangaea' / 'sample_20260629_163144_extents.json'

MODEL_NAMES = {
    'default-text-large': 'Llama3 70B',
    'default-text-large_recovered': 'Llama3 70B',
    'Qwen_Qwen3.6-35B-A3B-FP8': 'Qwen 3.6 35B',
    'Qwen3.6-35B-A3B-FP8': 'Qwen 3.6 35B',
    'Sehyo_Qwen3.5-122B-A10B-NVFP4': 'Qwen 3.5 122B',
    'Qwen_Qwen3.5-122B-A10B-NVFP4': 'Qwen 3.5 122B',
    'Qwen3.5-122B-A10B-NVFP4': 'Qwen 3.5 122B',
    'mistralai_Mistral-Small-3.2-24B-Instruct-2506': 'Mistral Small 3.2 24B',
    'mistralai_Devstral-Small-2-24B-Instruct-2512': 'Mistral Devstral 2 24B',
    'Devstral-Small-2-24B-Instruct-2512': 'Mistral Devstral 2 24B',
}

def display_name(run_folder, prefix):
    key = run_folder.replace(prefix, '', 1)
    return MODEL_NAMES.get(key, key)


In [2]:
def fix_dt(dt_str):
    '''Final format: YYYY-MM-DDTHH:MM:SS
    '''
    if not dt_str or not isinstance(dt_str, str):
        return None

    if dt_str[-1] == 'Z':
        dt_str = dt_str[:-1]

    if len(dt_str) == 10:
        dt_str = dt_str + 'T00:00:00'
    elif len(dt_str) == 16 and dt_str[10] == 'T' and dt_str[13] == ':':
        dt_str = dt_str + ':00'

    return dt_str


def read_technical_pred(path):
    '''DataFrame indexed like ``pd.read_json(..., orient='records')`` on an M3 file.

    Notebook 03 looks up ``df_pred.loc['min_lon', 'spatialCoverage']`` and
    ``df_pred.loc['from', 'temporalCoverage']``. Extracted files add a description
    and may set either coverage object to null, which makes that ``read_json`` call fail.
    '''
    with open(path) as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict) or payload.get('status') == 'skipped':
        return pd.DataFrame(columns=['spatialCoverage', 'temporalCoverage'])
    # M³, AutoDDG, and DataAtlas keep coverage at the top level. Single LLM nests it under metadata.
    if not isinstance(payload.get('spatialCoverage'), dict) and not isinstance(payload.get('temporalCoverage'), dict):
        metadata = payload.get('metadata')
        if isinstance(metadata, dict):
            payload = metadata
    frame = {}
    for column in ('spatialCoverage', 'temporalCoverage'):
        value = payload.get(column)
        if isinstance(value, dict):
            for key, item in value.items():
                frame.setdefault(key, {})[column] = item
    if not frame:
        return pd.DataFrame(columns=['spatialCoverage', 'temporalCoverage'])
    return pd.DataFrame.from_dict(frame, orient='index')


def _coverage_value(df_pred, key, column):
    if key not in df_pred.index or column not in df_pred.columns:
        return None
    value = df_pred.loc[key, column]
    if isinstance(value, (list, dict)):
        return None
    if pd.isna(value):
        return None
    return value


def locate_prediction(folder_pred, species_name):
    stem = species_name.replace(' ', '_')
    for name in (
        f'{stem}_inaturalist.json',
        f'{stem}_react.json',
        f'{stem}_react.skipped.json',
        f'{stem}_single_llm.json',
        f'{stem}_single_llm.skipped.json',
        f'{stem}.json',
    ):
        path = os.path.join(folder_pred, name)
        if os.path.exists(path):
            return path
    return None


## iNaturalist

`eval_spat_temp` is the notebook 03 scorer. Match counts are exact equality on the four bounds and the two timestamps. AutoDDG and DataAtlas are scored separately.

In [3]:
def eval_spat_temp(folder_pred, folder_data, notebook03_loader=False):
    base_cols = ['min_lon', 'max_lon', 'min_lat', 'max_lat', 'min_time', 'max_time']
    df_metadata = {
        x: []
        for x in ['id']
        + [f"{c}_annot" for c in base_cols]
        + [f"{c}_pred" for c in base_cols]
    }
    missing = []

    for f in os.listdir(folder_data):
        species_name = f.replace('.csv', '').replace('_', ' ')
        df_sel = pd.read_csv(os.path.join(folder_data, f))
        lon = df_sel['decimalLongitude']
        lat = df_sel['decimalLatitude']
        min_lon, max_lon = lon.min(), lon.max()
        min_lat, max_lat = lat.min(), lat.max()

        df_sel['eventDate_fixed'] = df_sel['eventDate'].map(fix_dt)
        df_sel['datetime'] = pd.to_datetime(df_sel['eventDate_fixed'], errors='coerce')
        assert df_sel['datetime'].isnull().sum() == 0, f"Some datetime values could not be parsed: {df_sel[df_sel['datetime'].isnull()]['eventDate'].tolist()}"
        min_time, max_time = df_sel['datetime'].min(), df_sel['datetime'].max()

        df_metadata['id'].append(species_name)
        df_metadata['min_lon_annot'].append(min_lon)
        df_metadata['max_lon_annot'].append(max_lon)
        df_metadata['min_lat_annot'].append(min_lat)
        df_metadata['max_lat_annot'].append(max_lat)
        df_metadata['min_time_annot'].append(min_time)
        df_metadata['max_time_annot'].append(max_time)

        pred_path = locate_prediction(folder_pred, species_name)
        if pred_path is None:
            missing.append(species_name)
            for col in base_cols:
                df_metadata[f'{col}_pred'].append(None)
            continue

        if notebook03_loader:
            df_pred = pd.read_json(pred_path, orient='records')
            df_metadata['min_lon_pred'].append(df_pred.loc['min_lon', 'spatialCoverage'])
            df_metadata['max_lon_pred'].append(df_pred.loc['max_lon', 'spatialCoverage'])
            df_metadata['min_lat_pred'].append(df_pred.loc['min_lat', 'spatialCoverage'])
            df_metadata['max_lat_pred'].append(df_pred.loc['max_lat', 'spatialCoverage'])
            df_metadata['min_time_pred'].append(fix_dt(df_pred.loc['from', 'temporalCoverage']) if 'from' in df_pred.index else None)
            df_metadata['max_time_pred'].append(fix_dt(df_pred.loc['to', 'temporalCoverage']) if 'to' in df_pred.index else None)
            continue

        df_pred = read_technical_pred(pred_path)
        df_metadata['min_lon_pred'].append(_coverage_value(df_pred, 'min_lon', 'spatialCoverage'))
        df_metadata['max_lon_pred'].append(_coverage_value(df_pred, 'max_lon', 'spatialCoverage'))
        df_metadata['min_lat_pred'].append(_coverage_value(df_pred, 'min_lat', 'spatialCoverage'))
        df_metadata['max_lat_pred'].append(_coverage_value(df_pred, 'max_lat', 'spatialCoverage'))
        df_metadata['min_time_pred'].append(fix_dt(_coverage_value(df_pred, 'from', 'temporalCoverage')))
        df_metadata['max_time_pred'].append(fix_dt(_coverage_value(df_pred, 'to', 'temporalCoverage')))

    df_metadata = pd.DataFrame(df_metadata)
    df_metadata['min_time_pred'] = pd.to_datetime(df_metadata['min_time_pred'], errors='coerce').dt.tz_localize(None)
    df_metadata['max_time_pred'] = pd.to_datetime(df_metadata['max_time_pred'], errors='coerce').dt.tz_localize(None)

    scores = {}
    for c in base_cols:
        scores[c] = float((df_metadata[f'{c}_pred'] == df_metadata[f'{c}_annot']).sum())

    scores['temp_average'] = float(np.mean([scores[c] for c in ['min_time', 'max_time']]))
    scores['spat_average'] = float(np.mean([scores[c] for c in ['min_lon', 'max_lon', 'min_lat', 'max_lat']]))
    scores['n_missing'] = len(missing)
    return df_metadata, scores, missing


In [4]:
inat_runs = sorted(p.name for p in (TECH_ROOT / 'inaturalist').iterdir() if p.is_dir() and p.name.startswith('autoddg_'))
df_results_inat = {x: [] for x in ['model', 'temporalCoverage', 'spatialCoverage', 'n_missing']}
inat_details = {}

for run in inat_runs:
    df_metadata, scores, missing = eval_spat_temp(TECH_ROOT / 'inaturalist' / run, INAT_DATA)
    model_name = display_name(run, 'autoddg_inat_')
    print(f"{model_name}: { {k: v for k, v in scores.items() if k != 'n_missing'} }")
    if missing:
        print(f"  missing extraction files ({len(missing)}): {missing}")
    inat_details[model_name] = df_metadata
    df_results_inat['model'].append(model_name)
    df_results_inat['temporalCoverage'].append(scores['temp_average'])
    df_results_inat['spatialCoverage'].append(scores['spat_average'])
    df_results_inat['n_missing'].append(scores['n_missing'])

df_results_inat = pd.DataFrame(df_results_inat)
df_results_inat['spatialCoverage'] = df_results_inat['spatialCoverage'].round(1)
df_results_inat['temporalCoverage'] = df_results_inat['temporalCoverage'].round(1)
df_results_inat = df_results_inat.rename(columns={
    'temporalCoverage': 'Temp. accuracy [%]',
    'spatialCoverage': 'Spat. accuracy [%]',
    'model': 'Model',
    'n_missing': 'Missing files',
})
df_results_inat = df_results_inat.set_index('Model').sort_index()
df_results_inat.to_csv(TECH_ROOT / 'df_results_autoddg_inat_spat_temp.csv')
df_results_inat

Qwen 3.6 35B: {'min_lon': 0.0, 'max_lon': 0.0, 'min_lat': 0.0, 'max_lat': 0.0, 'min_time': 0.0, 'max_time': 0.0, 'temp_average': 0.0, 'spat_average': 0.0}


Qwen 3.5 122B: {'min_lon': 0.0, 'max_lon': 0.0, 'min_lat': 0.0, 'max_lat': 0.0, 'min_time': 0.0, 'max_time': 0.0, 'temp_average': 0.0, 'spat_average': 0.0}


Llama3 70B: {'min_lon': 0.0, 'max_lon': 0.0, 'min_lat': 0.0, 'max_lat': 0.0, 'min_time': 0.0, 'max_time': 0.0, 'temp_average': 0.0, 'spat_average': 0.0}
  missing extraction files (1): ['acontia cretata']


Mistral Devstral 2 24B: {'min_lon': 0.0, 'max_lon': 0.0, 'min_lat': 0.0, 'max_lat': 0.0, 'min_time': 0.0, 'max_time': 0.0, 'temp_average': 0.0, 'spat_average': 0.0}
  missing extraction files (1): ['acianthus fornicatus']


Mistral Small 3.2 24B: {'min_lon': 0.0, 'max_lon': 0.0, 'min_lat': 0.0, 'max_lat': 0.0, 'min_time': 0.0, 'max_time': 0.0, 'temp_average': 0.0, 'spat_average': 0.0}


,Temp. accuracy [%],Spat. accuracy [%],Missing files
Model,,,
Llama3 70B,0.0,0.0,1
Mistral Devstral 2 24B,0.0,0.0,1
Mistral Small 3.2 24B,0.0,0.0,0
Qwen 3.5 122B,0.0,0.0,0
Qwen 3.6 35B,0.0,0.0,0


In [5]:
dataatlas_inat_runs = sorted(
    p.name for p in (TECH_ROOT / 'inaturalist').iterdir()
    if p.is_dir() and p.name.startswith('dataatlas_')
)
df_results_dataatlas_inat = {x: [] for x in ['model', 'temporalCoverage', 'spatialCoverage', 'n_missing']}
dataatlas_inat_details = {}

for run in dataatlas_inat_runs:
    df_metadata, scores, missing = eval_spat_temp(TECH_ROOT / 'inaturalist' / run, INAT_DATA)
    model_name = display_name(run, 'dataatlas_inat_')
    print(f"{model_name}: { {k: v for k, v in scores.items() if k != 'n_missing'} }")
    if missing:
        print(f"  missing extraction files ({len(missing)}): {missing}")
    dataatlas_inat_details[model_name] = df_metadata
    df_results_dataatlas_inat['model'].append(model_name)
    df_results_dataatlas_inat['temporalCoverage'].append(scores['temp_average'])
    df_results_dataatlas_inat['spatialCoverage'].append(scores['spat_average'])
    df_results_dataatlas_inat['n_missing'].append(scores['n_missing'])

df_results_dataatlas_inat = pd.DataFrame(df_results_dataatlas_inat)
df_results_dataatlas_inat['spatialCoverage'] = df_results_dataatlas_inat['spatialCoverage'].round(1)
df_results_dataatlas_inat['temporalCoverage'] = df_results_dataatlas_inat['temporalCoverage'].round(1)
df_results_dataatlas_inat = df_results_dataatlas_inat.rename(columns={
    'temporalCoverage': 'Temp. accuracy [%]',
    'spatialCoverage': 'Spat. accuracy [%]',
    'model': 'Model',
    'n_missing': 'Missing files',
})
df_results_dataatlas_inat = df_results_dataatlas_inat.set_index('Model').sort_index()
df_results_dataatlas_inat.to_csv(TECH_ROOT / 'df_results_dataatlas_inat_spat_temp.csv')
df_results_dataatlas_inat

Qwen 3.6 35B: {'min_lon': 0.0, 'max_lon': 0.0, 'min_lat': 0.0, 'max_lat': 0.0, 'min_time': 0.0, 'max_time': 0.0, 'temp_average': 0.0, 'spat_average': 0.0}
  missing extraction files (2): ['telicota colon', 'corvus torquatus']


Qwen 3.5 122B: {'min_lon': 0.0, 'max_lon': 0.0, 'min_lat': 0.0, 'max_lat': 0.0, 'min_time': 0.0, 'max_time': 0.0, 'temp_average': 0.0, 'spat_average': 0.0}
  missing extraction files (3): ['telicota colon', 'corvus torquatus', 'acianthus fornicatus']


Llama3 70B: {'min_lon': 0.0, 'max_lon': 0.0, 'min_lat': 0.0, 'max_lat': 0.0, 'min_time': 0.0, 'max_time': 0.0, 'temp_average': 0.0, 'spat_average': 0.0}
  missing extraction files (7): ['telicota colon', 'amyema preissii', 'bulla gouldiana', 'lepomis marginatus', 'corvus torquatus', 'physiphora alceae', 'mutinus caninus']


Mistral Devstral 2 24B: {'min_lon': 0.0, 'max_lon': 0.0, 'min_lat': 0.0, 'max_lat': 0.0, 'min_time': 0.0, 'max_time': 0.0, 'temp_average': 0.0, 'spat_average': 0.0}
  missing extraction files (4): ['telicota colon', 'acontia cretata', 'corvus torquatus', 'acianthus fornicatus']


Mistral Small 3.2 24B: {'min_lon': 0.0, 'max_lon': 0.0, 'min_lat': 0.0, 'max_lat': 0.0, 'min_time': 0.0, 'max_time': 0.0, 'temp_average': 0.0, 'spat_average': 0.0}
  missing extraction files (2): ['telicota colon', 'corvus torquatus']


,Temp. accuracy [%],Spat. accuracy [%],Missing files
Model,,,
Llama3 70B,0.0,0.0,7
Mistral Devstral 2 24B,0.0,0.0,4
Mistral Small 3.2 24B,0.0,0.0,2
Qwen 3.5 122B,0.0,0.0,3
Qwen 3.6 35B,0.0,0.0,2


## PANGAEA

Extents come from each dataset's `<md:extent>` block on the PANGAEA metadata index: `westBoundLongitude`, `eastBoundLongitude`, `southBoundLatitude`, `northBoundLatitude`, `minDateTime`, and `maxDateTime`. That is the metadata `PanDataSet.geometryextent` / `mintimeextent` / `maxtimeextent` reads in notebook 03. The parsed extents are cached next to the sample table.

In [6]:
def _extent_tag(block, tag):
    match = re.search(rf'<md:{tag}>(.*?)</md:{tag}>', block)
    return match.group(1) if match else None


def parse_pangaea_extent(xml):
    blocks = re.findall(r'<md:extent>(.*?)</md:extent>', xml, flags=re.S)
    geographic = next((b for b in blocks if 'westBoundLongitude' in b), '')
    temporal = next((b for b in blocks if 'minDateTime' in b), '')
    west = _extent_tag(geographic, 'westBoundLongitude')
    east = _extent_tag(geographic, 'eastBoundLongitude')
    south = _extent_tag(geographic, 'southBoundLatitude')
    north = _extent_tag(geographic, 'northBoundLatitude')
    if None in (west, east, south, north):
        spat_extent = None
    else:
        spat_extent = {
            'westBoundLongitude': float(west),
            'eastBoundLongitude': float(east),
            'southBoundLatitude': float(south),
            'northBoundLatitude': float(north),
        }
    tmin = _extent_tag(temporal, 'minDateTime')
    tmax = _extent_tag(temporal, 'maxDateTime')
    temp_extent = None if tmin is None or tmax is None else [tmin, tmax]
    return spat_extent, temp_extent


def load_pangaea_annotations():
    df_metadata_annot = pd.read_csv(PANGAEA_ANNOT)
    if PANGAEA_EXTENT_CACHE.exists():
        cached = json.loads(PANGAEA_EXTENT_CACHE.read_text())
    else:
        ids = [str(i) for i in df_metadata_annot['id'].tolist()]
        body = json.dumps({
            'size': len(ids),
            'query': {'ids': {'values': ids}},
            '_source': ['xml'],
        }).encode()
        request = urllib.request.Request(
            'https://ws.pangaea.de/es/pangaea/panmd/_search',
            data=body,
            headers={'Content-Type': 'application/json'},
        )
        with urllib.request.urlopen(request, timeout=120) as response:
            payload = json.load(response)
        cached = {}
        for hit in payload['hits']['hits']:
            spat_extent, temp_extent = parse_pangaea_extent(hit['_source']['xml'])
            cached[hit['_id']] = {'spat_extent': spat_extent, 'temp_extent': temp_extent}
        missing_ids = sorted(set(ids) - set(cached))
        if missing_ids:
            raise RuntimeError(f'PANGAEA metadata missing for {missing_ids}')
        PANGAEA_EXTENT_CACHE.write_text(json.dumps(cached))
        print(f'Cached extents for {len(cached)} datasets to {PANGAEA_EXTENT_CACHE.name}')

    df_metadata_annot['spat_extent'] = df_metadata_annot['id'].map(lambda i: cached[str(i)]['spat_extent'])
    df_metadata_annot['temp_extent'] = df_metadata_annot['id'].map(lambda i: cached[str(i)]['temp_extent'])
    print(
        f"{df_metadata_annot.temp_extent.isnull().sum()} datasets have no temporal extent. "
        f"{df_metadata_annot.spat_extent.isnull().sum()} datasets have no spatial extent."
    )
    return df_metadata_annot

df_metadata_annot = load_pangaea_annotations()


14 datasets have no temporal extent. 7 datasets have no spatial extent.


In [7]:
def eval_spat_temp_pangaea(folder_pred, df_metadata_annot, output_av_score_only=False):
    assert os.path.exists(folder_pred), f"Prediction folder does not exist: {folder_pred}"
    base_cols = ['min_lon', 'max_lon', 'min_lat', 'max_lat', 'min_time', 'max_time']
    df_metadata = {
        x: []
        for x in ['id']
        + [f"{c}_annot" for c in base_cols]
        + [f"{c}_pred" for c in base_cols]
    }

    ds_visited = set()
    for f in os.listdir(folder_pred):
        if not f.endswith('.json'):
            continue
        dataset_token = Path(f).stem.split('_')[0]
        if not dataset_token.isdigit():
            continue
        ds_name = int(dataset_token)
        assert ds_name not in ds_visited, f"Duplicate dataset {ds_name} found in predictions; skipping."
        ds_visited.add(ds_name)
        ind_ds = df_metadata_annot[df_metadata_annot.id == ds_name].index
        if len(ind_ds) == 0:
            print(f"Dataset {ds_name} not found in annotations; skipping.")
            continue
        assert len(ind_ds) == 1, f"Multiple entries found for dataset {ds_name} in annotations."

        df_metadata['id'].append(ds_name)
        spat_extent = df_metadata_annot.loc[ind_ds, 'spat_extent'].values[0]
        if spat_extent is not None:
            min_lat, max_lat = spat_extent['southBoundLatitude'], spat_extent['northBoundLatitude']
            min_lon, max_lon = spat_extent['westBoundLongitude'], spat_extent['eastBoundLongitude']
        else:
            min_lon, max_lon, min_lat, max_lat = None, None, None, None
        temp_extent = df_metadata_annot.loc[ind_ds, 'temp_extent'].values[0]
        if temp_extent is not None:
            min_time, max_time = temp_extent[0], temp_extent[1]
        else:
            min_time, max_time = None, None

        df_metadata['min_lon_annot'].append(min_lon)
        df_metadata['max_lon_annot'].append(max_lon)
        df_metadata['min_lat_annot'].append(min_lat)
        df_metadata['max_lat_annot'].append(max_lat)
        df_metadata['min_time_annot'].append(min_time)
        df_metadata['max_time_annot'].append(max_time)

        df_pred = read_technical_pred(os.path.join(folder_pred, f))
        if 'min_lon' in df_pred.index:
            df_metadata['min_lon_pred'].append(_coverage_value(df_pred, 'min_lon', 'spatialCoverage'))
            df_metadata['max_lon_pred'].append(_coverage_value(df_pred, 'max_lon', 'spatialCoverage'))
            df_metadata['min_lat_pred'].append(_coverage_value(df_pred, 'min_lat', 'spatialCoverage'))
            df_metadata['max_lat_pred'].append(_coverage_value(df_pred, 'max_lat', 'spatialCoverage'))
        else:
            df_metadata['min_lon_pred'].append(None)
            df_metadata['max_lon_pred'].append(None)
            df_metadata['min_lat_pred'].append(None)
            df_metadata['max_lat_pred'].append(None)

        df_metadata['min_time_pred'].append(fix_dt(_coverage_value(df_pred, 'from', 'temporalCoverage')))
        df_metadata['max_time_pred'].append(fix_dt(_coverage_value(df_pred, 'to', 'temporalCoverage')))

    ds_all = set(df_metadata_annot.id.values)
    ds_not_visited = ds_all - ds_visited
    print(f"Datasets not visited in predictions: {len(ds_not_visited)}")
    if ds_not_visited:
        for ds in ds_not_visited:
            df_metadata['min_lon_annot'].append(None)
            df_metadata['max_lon_annot'].append(None)
            df_metadata['min_lat_annot'].append(None)
            df_metadata['max_lat_annot'].append(None)
            df_metadata['min_time_annot'].append(None)
            df_metadata['max_time_annot'].append(None)
            df_metadata['min_lon_pred'].append(None)
            df_metadata['max_lon_pred'].append(None)
            df_metadata['min_lat_pred'].append(None)
            df_metadata['max_lat_pred'].append(None)
            df_metadata['min_time_pred'].append(None)
            df_metadata['max_time_pred'].append(None)
            df_metadata['id'].append(ds)

    for k, v in df_metadata.items():
        if len(v) != len(df_metadata['id']):
            raise ValueError(f"Length mismatch for key '{k}': expected {len(df_metadata['id'])}, got {len(v)}")
    df_metadata = pd.DataFrame(df_metadata)
    df_metadata['min_time_pred'] = pd.to_datetime(df_metadata['min_time_pred'].apply(lambda x: x[:19] if isinstance(x, str) else None), errors='coerce').dt.tz_localize(None)
    df_metadata['max_time_pred'] = pd.to_datetime(df_metadata['max_time_pred'].apply(lambda x: x[:19] if isinstance(x, str) else None), errors='coerce').dt.tz_localize(None)
    df_metadata['min_time_annot'] = pd.to_datetime(df_metadata['min_time_annot'], errors='coerce').dt.tz_localize(None)
    df_metadata['max_time_annot'] = pd.to_datetime(df_metadata['max_time_annot'], errors='coerce').dt.tz_localize(None)
    # Extracted dates can be year 0, for example 0000-01-01. Pandas parses that,
    # then .dt.date raises. Those values are not real dates, so they count as missing.
    for col in ['min_time_pred', 'max_time_pred', 'min_time_annot', 'max_time_annot']:
        years = df_metadata[col].dt.year
        df_metadata.loc[years.notna() & ~years.between(1, 9999), col] = pd.NaT

    for c in ['min_lat', 'max_lat', 'min_lon', 'max_lon']:
        df_metadata[f'{c}_pred'] = pd.to_numeric(df_metadata[f'{c}_pred'], errors='coerce')
        df_metadata[f'{c}_annot'] = pd.to_numeric(df_metadata[f'{c}_annot'], errors='coerce')

    df_metadata = df_metadata.sort_values(by='id').reset_index(drop=True)

    scores = {}
    for c in base_cols:
        inds_tp = pd.notna(df_metadata[f'{c}_annot']) & pd.notna(df_metadata[f'{c}_pred'])
        inds_fp = pd.isna(df_metadata[f'{c}_annot']) & pd.notna(df_metadata[f'{c}_pred'])
        inds_fn = pd.notna(df_metadata[f'{c}_annot']) & pd.isna(df_metadata[f'{c}_pred'])

        if 'time' in c:
            scores[c + '_accuracy'] = float((df_metadata[f'{c}_pred'].dt.date[inds_tp] == df_metadata[f'{c}_annot'].dt.date[inds_tp]).sum())
        else:
            scores[c + '_accuracy'] = float(np.sum(np.isclose(df_metadata[f'{c}_pred'][inds_tp], df_metadata[f'{c}_annot'][inds_tp], equal_nan=True)))
        scores[c + '_precision'] = float(np.sum(inds_tp)) / float(np.sum(inds_tp) + np.sum(inds_fp)) if (np.sum(inds_tp) + np.sum(inds_fp)) > 0 else None
        scores[c + '_recall'] = float(np.sum(inds_tp)) / float(np.sum(inds_tp) + np.sum(inds_fn)) if (np.sum(inds_tp) + np.sum(inds_fn)) > 0 else None

    def _average(values):
        numbers = [value for value in values if value is not None]
        return float(np.mean(numbers)) if numbers else None

    for m in ['accuracy', 'precision', 'recall']:
        scores[f'temp_{m}_average'] = _average([scores[c] for c in [f'min_time_{m}', f'max_time_{m}']])
        scores[f'spat_{m}_average'] = _average([scores[c] for c in [f'min_lon_{m}', f'max_lon_{m}', f'min_lat_{m}', f'max_lat_{m}']])

    if output_av_score_only:
        scores = {k: v for k, v in scores.items() if k.endswith('_average')}

    return df_metadata, scores


In [8]:
pangaea_runs = sorted(p.name for p in (TECH_ROOT / 'pangaea').iterdir() if p.is_dir() and p.name.startswith('autoddg_'))
scores_all = {}
pangaea_details = {}

for run in pangaea_runs:
    print(run)
    df_metadata, scores = eval_spat_temp_pangaea(TECH_ROOT / 'pangaea' / run, df_metadata_annot, output_av_score_only=True)
    pangaea_details[run] = df_metadata
    model_name = display_name(run, 'autoddg_pangaea_')
    if len(scores_all) == 0:
        scores_all['model'] = [model_name]
        for k, v in scores.items():
            scores_all[k] = [v]
    else:
        scores_all['model'].append(model_name)
        for k, v in scores.items():
            scores_all[k].append(v)

df_scores_all = pd.DataFrame(scores_all)
cols_temp = [c for c in df_scores_all.columns if c.startswith('temp_')]
cols_spat = [c for c in df_scores_all.columns if c.startswith('spat_')]
for c in cols_temp + cols_spat:
    if 'precision' in c or 'recall' in c:
        df_scores_all[c] = df_scores_all[c] * 100
    df_scores_all[c] = df_scores_all[c].round(1)
df_scores_all.columns = pd.MultiIndex.from_tuples([(c.split('_')[0], '_'.join(c.split('_')[1:])) for c in df_scores_all.columns])
df_scores_all = df_scores_all.sort_index(axis=1, level=[0, 1], ascending=[True, True])
df_scores_all = df_scores_all.rename(columns={'model': 'Model', 'spat': 'Spatial coverage', 'temp': 'Temporal coverage'}, level=0)
df_scores_all = df_scores_all.rename(columns={'accuracy_average': 'TP accuracy [%]', 'precision_average': 'Precision [%]', 'recall_average': 'Recall [%]'}, level=1)
df_scores_all = df_scores_all.sort_values(by=[('Model', '')]).reset_index(drop=True)
df_scores_all.to_csv(TECH_ROOT / 'df_results_autoddg_pangaea_spat_temp.csv', index=False)
df_scores_all

autoddg_pangaea_Qwen_Qwen3.6-35B-A3B-FP8
Datasets not visited in predictions: 0
autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4
Datasets not visited in predictions: 0
autoddg_pangaea_default-text-large
Datasets not visited in predictions: 0
autoddg_pangaea_mistralai_Devstral-Small-2-24B-Instruct-2512


Datasets not visited in predictions: 0
autoddg_pangaea_mistralai_Mistral-Small-3.2-24B-Instruct-2506
Datasets not visited in predictions: 3


Model Spatial coverage                           \
                           TP accuracy [%] Precision [%] Recall [%]   
0              Llama3 70B             38.2         100.0       88.2   
1  Mistral Devstral 2 24B             11.2         100.0       17.6   
2   Mistral Small 3.2 24B              8.0         100.0       13.8   
3           Qwen 3.5 122B              7.2         100.0       14.7   
4            Qwen 3.6 35B             10.0         100.0       29.4   

  Temporal coverage                           
    TP accuracy [%] Precision [%] Recall [%]  
0              28.5          90.7       72.1  
1              12.5          93.5       47.5  
2               8.0          87.2       29.3  
3              14.5          92.9       63.9  
4              16.5          95.3       67.2

In [9]:
# Best model per method on PANGAEA. Rank is spatial TP accuracy, then temporal TP accuracy,
# then precision and recall. M³ reads the pangaea_inat runs; AutoDDG and DataAtlas read
# the extracted technical metadata.

M3_ROOT = Path('/home/com3dian/Github/metadata_agent/outputs')
COVERAGE_COLUMNS = [
    ('Spatial coverage', 'TP accuracy [%]'),
    ('Spatial coverage', 'Precision [%]'),
    ('Spatial coverage', 'Recall [%]'),
    ('Temporal coverage', 'TP accuracy [%]'),
    ('Temporal coverage', 'Precision [%]'),
    ('Temporal coverage', 'Recall [%]'),
]
BEST_MODEL_RANK = COVERAGE_COLUMNS


def score_method_runs(run_dirs, prefix):
    scores_all = {}
    for run in sorted(path for path in run_dirs if path.is_dir() and any(path.glob('*.json'))):
        _, scores = eval_spat_temp_pangaea(run, df_metadata_annot, output_av_score_only=True)
        label_folder = run.parent.parent.name if run.name == 'pangaea' and run.parent.name == 'm3' else run.name
        label_prefix = 'm3_fast_all_' if label_folder.startswith('m3_fast_all_') else prefix
        model_name = display_name(label_folder, label_prefix)
        if not scores_all:
            scores_all['model'] = [model_name]
            for key, value in scores.items():
                scores_all[key] = [value]
        else:
            scores_all['model'].append(model_name)
            for key, value in scores.items():
                scores_all[key].append(value)
    df_scores = pd.DataFrame(scores_all)
    for column in [c for c in df_scores.columns if c.startswith(('temp_', 'spat_'))]:
        numeric = pd.to_numeric(df_scores[column], errors='coerce')
        if 'precision' in column or 'recall' in column:
            numeric = numeric * 100
        df_scores[column] = numeric.round(1)
    df_scores.columns = pd.MultiIndex.from_tuples(
        [(c.split('_')[0], '_'.join(c.split('_')[1:])) for c in df_scores.columns]
    )
    df_scores = df_scores.rename(
        columns={'model': 'Model', 'spat': 'Spatial coverage', 'temp': 'Temporal coverage'},
        level=0,
    )
    df_scores = df_scores.rename(
        columns={
            'accuracy_average': 'TP accuracy [%]',
            'precision_average': 'Precision [%]',
            'recall_average': 'Recall [%]',
        },
        level=1,
    )
    return df_scores


def best_model_row(df_scores):
    return df_scores.sort_values(BEST_MODEL_RANK, ascending=False).iloc[0]


def compare_best_models(method_specs):
    rows = []
    for method_name, run_dirs, prefix in method_specs:
        best = best_model_row(score_method_runs(run_dirs, prefix))
        row = {('Method', ''): method_name, ('Best model', ''): best[('Model', '')]}
        for column in COVERAGE_COLUMNS:
            row[column] = best[column]
        rows.append(row)
    comparison = pd.DataFrame(rows)
    return comparison[[('Method', ''), ('Best model', '')] + COVERAGE_COLUMNS]


def print_coverage_latex(df):
    latex_df = df.copy()
    latex_df[('Method', '')] = latex_df[('Method', '')].replace({
        'M³ fast': r'M$^3$ fast',
        'M³': r'M$^3$',
    })
    latex = latex_df.to_latex(
        index=False,
        float_format='%.1f',
        escape=False,
        caption='Best model per method on PANGAEA spatial and temporal coverage. The best model is the highest spatial true-positive accuracy, then temporal true-positive accuracy.',
        label='tab:best_method_spat_temp',
    )
    print(latex)
    return latex


method_comparison = compare_best_models([
    ('M³', M3_ROOT.glob('pangaea_inat_*'), 'pangaea_inat_'),
    ('M³ fast', (path / 'm3' / 'pangaea' for path in M3_ROOT.glob('m3_fast_all_*')), 'm3_fast_all_'),
    ('AutoDDG', (TECH_ROOT / 'pangaea').glob('autoddg_*'), 'autoddg_pangaea_'),
    ('DataAtlas', (TECH_ROOT / 'pangaea').glob('dataatlas_*'), 'dataatlas_pangaea_'),
    ('Single LLM', M3_ROOT.glob('single_llm_pangaea_*'), 'single_llm_pangaea_'),
    ('ReAct', (p for p in M3_ROOT.glob('react_pangaea_*') if 'smoke' not in p.name), 'react_pangaea_'),
])
print_coverage_latex(method_comparison)
method_comparison

Datasets not visited in predictions: 7
Datasets not visited in predictions: 9


Datasets not visited in predictions: 8
Datasets not visited in predictions: 8


Datasets not visited in predictions: 0
Datasets not visited in predictions: 0


Datasets not visited in predictions: 0
Datasets not visited in predictions: 0


Datasets not visited in predictions: 0
Datasets not visited in predictions: 0


Datasets not visited in predictions: 0
Datasets not visited in predictions: 0


... (5 outputs left out)

Datasets not visited in predictions: 0


Datasets not visited in predictions: 0
Datasets not visited in predictions: 0
Datasets not visited in predictions: 2


\begin{table}
\caption{Best model per method on PANGAEA spatial and temporal coverage. The best model is the highest spatial true-positive accuracy, then temporal true-positive accuracy.}
\label{tab:best_method_spat_temp}
\begin{tabular}{llrrrrrr}
\toprule
('Method', '') & ('Best model', '') & ('Spatial coverage', 'TP accuracy [%]') & ('Spatial coverage', 'Precision [%]') & ('Spatial coverage', 'Recall [%]') & ('Temporal coverage', 'TP accuracy [%]') & ('Temporal coverage', 'Precision [%]') & ('Temporal coverage', 'Recall [%]') \\
\midrule
M$^3$ & Llama3 70B & 60.5 & 100.0 & 100.0 & 53.0 & 91.9 & 96.6 \\
M$^3$ fast & Llama3 70B & 60.5 & 100.0 & 97.1 & 53.0 & 95.0 & 93.4 \\
AutoDDG & Llama3 70B & 38.2 & 100.0 & 88.2 & 28.5 & 90.7 & 72.1 \\
DataAtlas & Llama3 70B & 0.0 & NaN & 0.0 & 18.5 & 100.0 & 69.2 \\
Single LLM & Llama3 70B & 53.0 & 100.0 & 89.7 & 46.5 & 91.2 & 85.2 \\
ReAct & Llama3 70B & 60.0 & 100.0 & 97.1 & 51.5 & 95.0 & 93.4 \\
\bottomrule
\end{tabular}
\end{table}



,"(Method, )","(Best model, )","(Spatial coverage, TP accuracy [%])","(Spatial coverage, Precision [%])","(Spatial coverage, Recall [%])","(Temporal coverage, TP accuracy [%])","(Temporal coverage, Precision [%])","(Temporal coverage, Recall [%])"
0,M³,Llama3 70B,60.5,100.0,100.0,53.0,91.9,96.6
1,M³ fast,Llama3 70B,60.5,100.0,97.1,53.0,95.0,93.4
2,AutoDDG,Llama3 70B,38.2,100.0,88.2,28.5,90.7,72.1
3,DataAtlas,Llama3 70B,0.0,NaN,0.0,18.5,100.0,69.2
4,Single LLM,Llama3 70B,53.0,100.0,89.7,46.5,91.2,85.2
5,ReAct,Llama3 70B,60.0,100.0,97.1,51.5,95.0,93.4


In [10]:
def print_best_pangaea_latex(df):
    latex_df = df.copy()
    latex_df.columns = [
        'Method',
        'Best model',
        'Spatial TP accuracy [%]',
        'Spatial precision [%]',
        'Spatial recall [%]',
        'Temporal TP accuracy [%]',
        'Temporal precision [%]',
        'Temporal recall [%]',
    ]
    latex_df['Method'] = latex_df['Method'].replace({
        'M³ fast': 'M3FASTPLACEHOLDER',
        'M³': 'M3PLACEHOLDER',
    })
    latex = latex_df.to_latex(
        index=False,
        escape=True,
        float_format='%.1f',
        na_rep='--',
        column_format='llrrrrrr',
        caption=(
            'Best model per method on PANGAEA spatial and temporal coverage. '
            'The best model is the highest spatial true-positive accuracy, then spatial precision, '
            'spatial recall, temporal true-positive accuracy, temporal precision, and temporal recall.'
        ),
        label='tab:best_method_spat_temp',
    )
    old_header = (
        'Method & Best model & Spatial TP accuracy [\\%] & Spatial precision [\\%] & Spatial recall [\\%] & '
        'Temporal TP accuracy [\\%] & Temporal precision [\\%] & Temporal recall [\\%] \\\\\n'
    )
    new_header = (
        'Method & Best model & \\multicolumn{3}{c}{Spatial coverage} & \\multicolumn{3}{c}{Temporal coverage} \\\\\n'
        '\\cmidrule(lr){3-5} \\cmidrule(lr){6-8}\n'
        ' &  & TP accuracy [\\%] & Precision [\\%] & Recall [\\%] & TP accuracy [\\%] & Precision [\\%] & Recall [\\%] \\\\\n'
    )
    latex = latex.replace(old_header, new_header)
    latex = latex.replace('M3FASTPLACEHOLDER', r'M$^3$ fast')
    latex = latex.replace('M3PLACEHOLDER', r'M$^3$')
    tex_path = TECH_ROOT / 'df_results_best_method_spat_temp.tex'
    tex_path.write_text(latex, encoding='utf-8')
    print(latex)
    print(f'Saved {tex_path}')
    return latex


print_best_pangaea_latex(method_comparison)


\begin{table}
\caption{Best model per method on PANGAEA spatial and temporal coverage. The best model is the highest spatial true-positive accuracy, then spatial precision, spatial recall, temporal true-positive accuracy, temporal precision, and temporal recall.}
\label{tab:best_method_spat_temp}
\begin{tabular}{llrrrrrr}
\toprule
Method & Best model & \multicolumn{3}{c}{Spatial coverage} & \multicolumn{3}{c}{Temporal coverage} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
 &  & TP accuracy [\%] & Precision [\%] & Recall [\%] & TP accuracy [\%] & Precision [\%] & Recall [\%] \\
\midrule
M$^3$ & Llama3 70B & 60.5 & 100.0 & 100.0 & 53.0 & 91.9 & 96.6 \\
M$^3$ fast & Llama3 70B & 60.5 & 100.0 & 97.1 & 53.0 & 95.0 & 93.4 \\
AutoDDG & Llama3 70B & 38.2 & 100.0 & 88.2 & 28.5 & 90.7 & 72.1 \\
DataAtlas & Llama3 70B & 0.0 & -- & 0.0 & 18.5 & 100.0 & 69.2 \\
Single LLM & Llama3 70B & 53.0 & 100.0 & 89.7 & 46.5 & 91.2 & 85.2 \\
ReAct & Llama3 70B & 60.0 & 100.0 & 97.1 & 51.5 & 95.0 & 93.4 \\
\bottomr

'\\begin{table}\n\\caption{Best model per method on PANGAEA spatial and temporal coverage. The best model is the highest spatial true-positive accuracy, then spatial precision, spatial recall, temporal true-positive accuracy, temporal precision, and temporal recall.}\n\\label{tab:best_method_spat_temp}\n\\begin{tabular}{llrrrrrr}\n\\toprule\nMethod & Best model & \\multicolumn{3}{c}{Spatial coverage} & \\multicolumn{3}{c}{Temporal coverage} \\\\\n\\cmidrule(lr){3-5} \\cmidrule(lr){6-8}\n &  & TP accuracy [\\%] & Precision [\\%] & Recall [\\%] & TP accuracy [\\%] & Precision [\\%] & Recall [\\%] \\\\\n\\midrule\nM$^3$ & Llama3 70B & 60.5 & 100.0 & 100.0 & 53.0 & 91.9 & 96.6 \\\\\nM$^3$ fast & Llama3 70B & 60.5 & 100.0 & 97.1 & 53.0 & 95.0 & 93.4 \\\\\nAutoDDG & Llama3 70B & 38.2 & 100.0 & 88.2 & 28.5 & 90.7 & 72.1 \\\\\nDataAtlas & Llama3 70B & 0.0 & -- & 0.0 & 18.5 & 100.0 & 69.2 \\\\\nSingle LLM & Llama3 70B & 53.0 & 100.0 & 89.7 & 46.5 & 91.2 & 85.2 \\\\\nReAct & Llama3 70B & 60.0 & 

In [11]:
def all_technical_models(method_specs):
    model_order = [
        'Qwen 3.6 35B',
        'Qwen 3.5 122B',
        'Llama3 70B',
        'Mistral Devstral 2 24B',
        'Mistral Small 3.2 24B',
    ]
    frames = []
    for method_name, run_dirs, prefix in method_specs:
        frame = score_method_runs(run_dirs, prefix).copy()
        frame.insert(0, ('Method', ''), method_name)
        frame[('Model', '')] = pd.Categorical(frame[('Model', '')], model_order, ordered=True)
        frames.append(frame.sort_values(('Model', '')).reset_index(drop=True))
    table = pd.concat(frames, ignore_index=True)
    return table[[('Method', ''), ('Model', '')] + COVERAGE_COLUMNS]


def print_all_technical_latex(df):
    latex_df = df.copy()
    latex_df[('Method', '')] = latex_df[('Method', '')].replace({
        'M³ fast': 'M3FASTPLACEHOLDER',
        'M³': 'M3PLACEHOLDER',
    })
    grouped = latex_df.set_index([('Method', ''), ('Model', '')])
    grouped.index.names = ['Method', 'Model']
    latex = grouped.to_latex(
        index=True,
        escape=True,
        multirow=True,
        float_format='%.1f',
        na_rep='--',
        caption=r'Spatial and temporal coverage for every model of M$^3$, M$^3$ fast, AutoDDG, DataAtlas, Single LLM, and ReAct on PANGAEA.',
        label='tab:all_models_spat_temp',
    )
    old_header = (
        ' &  & \\multicolumn{3}{r}{Spatial coverage} & \\multicolumn{3}{r}{Temporal coverage} \\\\\n'
        ' &  & TP accuracy [\\%] & Precision [\\%] & Recall [\\%] & TP accuracy [\\%] & Precision [\\%] & Recall [\\%] \\\\\n'
        'Method & Model &  &  &  &  &  &  \\\\\n'
    )
    new_header = (
        'Method & Model & \\multicolumn{3}{c}{Spatial coverage} & \\multicolumn{3}{c}{Temporal coverage} \\\\\n'
        '\\cmidrule(lr){3-5} \\cmidrule(lr){6-8}\n'
        ' &  & TP accuracy [\\%] & Precision [\\%] & Recall [\\%] & TP accuracy [\\%] & Precision [\\%] & Recall [\\%] \\\\\n'
    )
    latex = latex.replace(old_header, new_header)
    latex = latex.replace(r'\cline{1-8}', r'\midrule')
    latex = latex.replace('\\midrule\n\\bottomrule', '\\bottomrule')
    latex = latex.replace('M3FASTPLACEHOLDER', r'M$^3$ fast')
    latex = latex.replace('M3PLACEHOLDER', r'M$^3$')
    print(latex)
    return latex


all_technical = all_technical_models([
    ('M³', M3_ROOT.glob('pangaea_inat_*'), 'pangaea_inat_'),
    ('M³ fast', (path / 'm3' / 'pangaea' for path in M3_ROOT.glob('m3_fast_all_*')), 'm3_fast_all_'),
    ('AutoDDG', (TECH_ROOT / 'pangaea').glob('autoddg_*'), 'autoddg_pangaea_'),
    ('DataAtlas', (TECH_ROOT / 'pangaea').glob('dataatlas_*'), 'dataatlas_pangaea_'),
    ('Single LLM', M3_ROOT.glob('single_llm_pangaea_*'), 'single_llm_pangaea_'),
    ('ReAct', (p for p in M3_ROOT.glob('react_pangaea_*') if 'smoke' not in p.name), 'react_pangaea_'),
])
print_all_technical_latex(all_technical)
all_technical.set_index([('Method', ''), ('Model', '')])

Datasets not visited in predictions: 7
Datasets not visited in predictions: 9
Datasets not visited in predictions: 8


Datasets not visited in predictions: 8


Datasets not visited in predictions: 0
Datasets not visited in predictions: 0
Datasets not visited in predictions: 0


Datasets not visited in predictions: 0


Datasets not visited in predictions: 0
Datasets not visited in predictions: 0
Datasets not visited in predictions: 0


Datasets not visited in predictions: 0


... (4 outputs left out)

Datasets not visited in predictions: 6
Datasets not visited in predictions: 5
Datasets not visited in predictions: 0


Datasets not visited in predictions: 0


Datasets not visited in predictions: 0
Datasets not visited in predictions: 0
Datasets not visited in predictions: 2
\begin{table}
\caption{Spatial and temporal coverage for every model of M$^3$, M$^3$ fast, AutoDDG, DataAtlas, Single LLM, and ReAct on PANGAEA.}
\label{tab:all_models_spat_temp}
\begin{tabular}{llrrrrrr}
\toprule
Method & Model & \multicolumn{3}{c}{Spatial coverage} & \multicolumn{3}{c}{Temporal coverage} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
 &  & TP accuracy [\%] & Precision [\%] & Recall [\%] & TP accuracy [\%] & Precision [\%] & Recall [\%] \\
\midrule
\multirow[t]{4}{*}{M$^3$} & Qwen 3.6 35B & 60.5 & 97.1 & 100.0 & 53.0 & 86.4 & 96.6 \\
 & Qwen 3.5 122B & 60.5 & 100.0 & 100.0 & 49.5 & 95.0 & 96.6 \\
 & Llama3 70B & 60.5 & 100.0 & 100.0 & 53.0 & 91.9 & 96.6 \\
 & Mistral Devstral 2 24B & 59.5 & 98.5 & 100.0 & 51.5 & 90.6 & 98.3 \\
\midrule
\multirow[t]{4}{*}{M$^3$ fast} & Qwen 3.6 35B & 59.5 & 97.1 & 97.1 & 49.0 & 96.5 & 90.2 \\
 & Qwen 3.5 122B & 59.5 & 100.0 & 

Spatial coverage                           \
                                   TP accuracy [%] Precision [%] Recall [%]   
(Method, ) (Model, )                                                          
M³         Qwen 3.6 35B                       60.5          97.1      100.0   
           Qwen 3.5 122B                      60.5         100.0      100.0   
           Llama3 70B                         60.5         100.0      100.0   
           Mistral Devstral 2 24B             59.5          98.5      100.0   
M³ fast    Qwen 3.6 35B                       59.5          97.1       97.1   
           Qwen 3.5 122B                      59.5         100.0       95.6   
           Llama3 70B                         60.5         100.0       97.1   
           Mistral Devstral 2 24B             60.0          97.0       95.6   
AutoDDG    Qwen 3.6 35B                       10.0         100.0       29.4   
           Qwen 3.5 122B                       7.2         100.0       14.7   
           Llama3 70B                         38.2         100.0       88.2   
           Mistral Devstral 2 24B             11.2         100.0       17.6   
           Mistral Small 3.2 24B               8.0         100.0       13.8   
DataAtlas  Qwen 3.6 35B                        0.0           NaN        0.0   
           Qwen 3.5 122B                       0.0           NaN        0.0   
           Llama3 70B                          0.0           NaN        0.0   
           Mistral Devstral 2 24B              0.0           NaN        0.0   
           Mistral Small 3.2 24B               0.0           NaN        0.0   
Single LLM Qwen 3.6 35B                       48.5          98.2       88.7   
           Qwen 3.5 122B                      48.5          96.6       88.9   
           Llama3 70B                         53.0         100.0       89.7   
           Mistral Devstral 2 24B             51.8         100.0       89.2   
           Mistral Small 3.2 24B              49.5         100.0       87.7   
ReAct      Qwen 3.6 35B                       59.5          98.5       95.6   
           Qwen 3.5 122B                      59.5          97.0       95.6   
           Llama3 70B                         60.0         100.0       97.1   
           Mistral Devstral 2 24B             57.5          98.4       92.6   
           Mistral Small 3.2 24B              58.5          98.5       97.0   

                                  Temporal coverage                           
                                    TP accuracy [%] Precision [%] Recall [%]  
(Method, ) (Model, )                                                          
M³         Qwen 3.6 35B                        53.0          86.4       96.6  
           Qwen 3.5 122B                       49.5          95.0       96.6  
           Llama3 70B                          53.0          91.9       96.6  
           Mistral Devstral 2 24B              51.5          90.6       98.3  
M³ fast    Qwen 3.6 35B                        49.0          96.5       90.2  
           Qwen 3.5 122B                       49.5          96.6       91.8  
           Llama3 70B                          53.0          95.0       93.4  
           Mistral Devstral 2 24B              53.0          92.1       95.1  
AutoDDG    Qwen 3.6 35B                        16.5          95.3       67.2  
           Qwen 3.5 122B                       14.5          92.9       63.9  
           Llama3 70B                          28.5          90.7       72.1  
           Mistral Devstral 2 24B              12.5          93.5       47.5  
           Mistral Small 3.2 24B                8.0          87.2       29.3  
DataAtlas  Qwen 3.6 35B                        18.5         100.0       56.7  
           Qwen 3.5 122B                       17.5         100.0       70.0  
           Llama3 70B                          18.5         100.0       69.2  
           Mistral Devstral 2 24B              14.5         100.0       51.9  
    

In [12]:
INAT_MODEL_ORDER = [
    'Llama3 70B',
    'Mistral Devstral 2 24B',
    'Mistral Small 3.2 24B',
    'Qwen 3.5 122B',
    'Qwen 3.6 35B',
]


def inat_accuracy_table(run_dirs, prefix):
    rows = []
    for run in sorted(path for path in run_dirs if path.is_dir() and any(path.glob('*.json'))):
        _, scores, _ = eval_spat_temp(run, INAT_DATA, notebook03_loader=(prefix == 'inat_'))
        label_folder = run.parent.parent.name if run.name == 'inaturalist' and run.parent.name == 'm3' else run.name
        label_prefix = 'm3_fast_all_' if label_folder.startswith('m3_fast_all_') else prefix
        rows.append({
            'Model': display_name(label_folder, label_prefix),
            'Temp. accuracy [%]': round(float(scores['temp_average']), 1),
            'Spat. accuracy [%]': round(float(scores['spat_average']), 1),
        })
    frame = pd.DataFrame(rows)
    frame['Model'] = pd.Categorical(frame['Model'], INAT_MODEL_ORDER, ordered=True)
    return frame.sort_values('Model').reset_index(drop=True)


def all_inat_methods():
    specs = [
        ('M³', (REPO / 'data' / 'predictions').glob('inat_*'), 'inat_'),
        ('M³ fast', (path / 'm3' / 'inaturalist' for path in M3_ROOT.glob('m3_fast_all_*')), 'm3_fast_all_'),
        ('AutoDDG', (TECH_ROOT / 'inaturalist').glob('autoddg_*'), 'autoddg_inat_'),
        ('DataAtlas', (TECH_ROOT / 'inaturalist').glob('dataatlas_*'), 'dataatlas_inat_'),
        ('Single LLM', M3_ROOT.glob('single_llm_inaturalist_*'), 'single_llm_inaturalist_'),
        ('ReAct', M3_ROOT.glob('react_inaturalist_*'), 'react_inaturalist_'),
    ]
    frames = []
    for method_name, run_dirs, prefix in specs:
        frame = inat_accuracy_table(run_dirs, prefix)
        frame.insert(0, 'Method', method_name)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def print_all_inat_latex(df):
    latex_df = df.copy()
    latex_df['Method'] = latex_df['Method'].replace({
        'M³ fast': 'M3FASTPLACEHOLDER',
        'M³': 'M3PLACEHOLDER',
    })
    grouped = latex_df.set_index(['Method', 'Model'])
    latex = grouped.to_latex(
        index=True,
        escape=True,
        multirow=True,
        float_format='%.1f',
        column_format='llcc',
        caption=r'Spatial and temporal accuracy for every model of M$^3$, M$^3$ fast, AutoDDG, DataAtlas, Single LLM, and ReAct on iNaturalist. A field is correct only when it equals the observation bound. With 100 species, the match count is the accuracy in percent.',
        label='tab:all_models_inat_spat_temp',
    )
    old_header = (
        ' &  & Temp. accuracy [\\%] & Spat. accuracy [\\%] \\\\\n'
        'Method & Model &  &  \\\\\n'
    )
    new_header = 'Method & Model & Temp. accuracy [\\%] & Spat. accuracy [\\%] \\\\\n'
    latex = latex.replace(old_header, new_header)
    latex = latex.replace(r'\cline{1-4}', r'\midrule')
    latex = latex.replace('\\midrule\n\\bottomrule', '\\bottomrule')
    latex = latex.replace('M3FASTPLACEHOLDER', r'M$^3$ fast')
    latex = latex.replace('M3PLACEHOLDER', r'M$^3$')
    tex_path = TECH_ROOT / 'df_results_all_methods_inat_spat_temp.tex'
    tex_path.write_text(latex, encoding='utf-8')
    print(latex)
    print(f'Saved {tex_path}')
    return latex


all_inat = all_inat_methods()
print_all_inat_latex(all_inat)
all_inat.set_index(['Method', 'Model'])

\begin{table}
\caption{Spatial and temporal accuracy for every model of M$^3$, M$^3$ fast, AutoDDG, DataAtlas, Single LLM, and ReAct on iNaturalist. A field is correct only when it equals the observation bound. With 100 species, the match count is the accuracy in percent.}
\label{tab:all_models_inat_spat_temp}
\begin{tabular}{llcc}
\toprule
Method & Model & Temp. accuracy [\%] & Spat. accuracy [\%] \\
\midrule
\multirow[t]{4}{*}{M$^3$} & Llama3 70B & 81.0 & 99.0 \\
 & Mistral Devstral 2 24B & 69.5 & 99.2 \\
 & Qwen 3.5 122B & 76.0 & 99.2 \\
 & Qwen 3.6 35B & 87.0 & 95.2 \\
\midrule
\multirow[t]{4}{*}{M$^3$ fast} & Llama3 70B & 78.5 & 100.0 \\
 & Mistral Devstral 2 24B & 96.0 & 96.0 \\
 & Qwen 3.5 122B & 96.0 & 96.0 \\
 & Qwen 3.6 35B & 100.0 & 100.0 \\
\midrule
\multirow[t]{5}{*}{AutoDDG} & Llama3 70B & 0.0 & 0.0 \\
 & Mistral Devstral 2 24B & 0.0 & 0.0 \\
 & Mistral Small 3.2 24B & 0.0 & 0.0 \\
 & Qwen 3.5 122B & 0.0 & 0.0 \\
 & Qwen 3.6 35B & 0.0 & 0.0 \\
\midrule
\multirow[t]{5}{*}{

Temp. accuracy [%]  Spat. accuracy [%]
Method     Model                                                         
M³         Llama3 70B                            81.0                99.0
           Mistral Devstral 2 24B                69.5                99.2
           Qwen 3.5 122B                         76.0                99.2
           Qwen 3.6 35B                          87.0                95.2
M³ fast    Llama3 70B                            78.5               100.0
           Mistral Devstral 2 24B                96.0                96.0
           Qwen 3.5 122B                         96.0                96.0
           Qwen 3.6 35B                         100.0               100.0
AutoDDG    Llama3 70B                             0.0                 0.0
           Mistral Devstral 2 24B                 0.0                 0.0
           Mistral Small 3.2 24B                  0.0                 0.0
           Qwen 3.5 122B                          0.0                 0.0
           Qwen 3.6 35B                           0.0                 0.0
DataAtlas  Llama3 70B                             0.0                 0.0
           Mistral Devstral 2 24B                 0.0                 0.0
           Mistral Small 3.2 24B                  0.0                 0.0
           Qwen 3.5 122B                          0.0                 0.0
           Qwen 3.6 35B                           0.0                 0.0
Single LLM Llama3 70B                             0.0                 0.0
           Mistral Devstral 2 24B                 0.0                 0.0
           Mistral Small 3.2 24B                  0.0                 0.0
           Qwen 3.5 122B                          0.0                 0.0
           Qwen 3.6 35B                           0.0                 0.0
ReAct      Llama3 70B                            92.0               100.0
           Mistral Devstral 2 24B                18.5               100.0
           Mistral Small 3.2 24B                 18.0                99.8
           Qwen 3.5 122B                         97.0                99.0
           Qwen 3.6 35B                          99.0                99.0

In [13]:
# Best model per method on iNaturalist. Rank is the mean of temporal and spatial
# accuracy. Equal means keep the earlier row, which is the model order of the
# every-model table.

def best_inat_model_by_mean(df):
    ranked = df.copy()
    ranked['Mean accuracy [%]'] = ranked[['Temp. accuracy [%]', 'Spat. accuracy [%]']].mean(axis=1)
    rows = []
    for _, group in ranked.groupby('Method', sort=False):
        best = group.sort_values(
            ['Mean accuracy [%]', 'Temp. accuracy [%]', 'Spat. accuracy [%]'],
            ascending=False,
            kind='mergesort',
        ).iloc[0]
        rows.append(best)
    best_table = pd.DataFrame(rows).reset_index(drop=True)
    best_table['Mean accuracy [%]'] = best_table['Mean accuracy [%]'].round(1)
    return best_table.rename(columns={'Model': 'Best model'})[[
        'Method',
        'Best model',
        'Temp. accuracy [%]',
        'Spat. accuracy [%]',
        'Mean accuracy [%]',
    ]]


def print_best_inat_latex(df):
    latex_df = df.copy()
    latex_df['Method'] = latex_df['Method'].replace({
        'M³ fast': 'M3FASTPLACEHOLDER',
        'M³': 'M3PLACEHOLDER',
    })
    latex = latex_df.to_latex(
        index=False,
        escape=True,
        float_format='%.1f',
        column_format='llccc',
        caption=(
            r'Best model per method on iNaturalist spatial and temporal accuracy. '
            r'The best model is the highest mean of the two accuracies. '
            r'Equal means keep the model that appears first in the every-model table.'
        ),
        label='tab:best_method_inat_spat_temp',
    )
    latex = latex.replace('M3FASTPLACEHOLDER', r'M$^3$ fast')
    latex = latex.replace('M3PLACEHOLDER', r'M$^3$')
    tex_path = TECH_ROOT / 'df_results_best_method_inat_spat_temp.tex'
    tex_path.write_text(latex, encoding='utf-8')
    print(latex)
    print(f'Saved {tex_path}')
    return latex


best_inat = best_inat_model_by_mean(all_inat)
print_best_inat_latex(best_inat)
best_inat


\begin{table}
\caption{Best model per method on iNaturalist spatial and temporal accuracy. The best model is the highest mean of the two accuracies. Equal means keep the model that appears first in the every-model table.}
\label{tab:best_method_inat_spat_temp}
\begin{tabular}{llccc}
\toprule
Method & Best model & Temp. accuracy [\%] & Spat. accuracy [\%] & Mean accuracy [\%] \\
\midrule
M$^3$ & Qwen 3.6 35B & 87.0 & 95.2 & 91.1 \\
M$^3$ fast & Qwen 3.6 35B & 100.0 & 100.0 & 100.0 \\
AutoDDG & Llama3 70B & 0.0 & 0.0 & 0.0 \\
DataAtlas & Llama3 70B & 0.0 & 0.0 & 0.0 \\
Single LLM & Llama3 70B & 0.0 & 0.0 & 0.0 \\
ReAct & Qwen 3.6 35B & 99.0 & 99.0 & 99.0 \\
\bottomrule
\end{tabular}
\end{table}

Saved /home/com3dian/Github/process_data_metadata_agent/outputs/technical_from_descriptions/df_results_best_method_inat_spat_temp.tex


,Method,Best model,Temp. accuracy [%],Spat. accuracy [%],Mean accuracy [%]
0,M³,Qwen 3.6 35B,87.0,95.2,91.1
1,M³ fast,Qwen 3.6 35B,100.0,100.0,100.0
2,AutoDDG,Llama3 70B,0.0,0.0,0.0
3,DataAtlas,Llama3 70B,0.0,0.0,0.0
4,Single LLM,Llama3 70B,0.0,0.0,0.0
5,ReAct,Qwen 3.6 35B,99.0,99.0,99.0


In [14]:
def temporal_year_accuracy(df_metadata):
    min_year = pd.to_datetime(df_metadata['min_time_annot'], errors='coerce').dt.year
    max_year = pd.to_datetime(df_metadata['max_time_annot'], errors='coerce').dt.year
    min_year_pred = pd.to_datetime(df_metadata['min_time_pred'], errors='coerce').dt.year
    max_year_pred = pd.to_datetime(df_metadata['max_time_pred'], errors='coerce').dt.year
    min_matches = float((min_year_pred == min_year).sum())
    max_matches = float((max_year_pred == max_year).sum())
    return (min_matches + max_matches) / 2


def nearest_integer(values):
    numeric = pd.to_numeric(values, errors='coerce')
    return np.sign(numeric) * np.floor(np.abs(numeric) + 0.5)


def spatial_integer_accuracy(df_metadata):
    matches = []
    for column in ['min_lat', 'max_lat', 'min_lon', 'max_lon']:
        pred = nearest_integer(df_metadata[f'{column}_pred'])
        annot = nearest_integer(df_metadata[f'{column}_annot'])
        matches.append(float((pred == annot).sum()))
    return float(np.mean(matches))


def inat_year_table(run_dirs, prefix):
    rows = []
    for run in sorted(path for path in run_dirs if path.is_dir() and any(path.glob('*.json'))):
        df_metadata, scores, _ = eval_spat_temp(run, INAT_DATA, notebook03_loader=(prefix == 'inat_'))
        rows.append({
            'Model': display_name(run.name, prefix),
            'Temp. year accuracy [%]': round(temporal_year_accuracy(df_metadata), 1),
            'Spat. integer accuracy [%]': round(spatial_integer_accuracy(df_metadata), 1),
            'Spat. accuracy [%]': round(float(scores['spat_average']), 1),
        })
    frame = pd.DataFrame(rows)
    frame['Model'] = pd.Categorical(frame['Model'], INAT_MODEL_ORDER, ordered=True)
    return frame.sort_values('Model').reset_index(drop=True)


def all_inat_year_methods():
    specs = [
        ('M³', (REPO / 'data' / 'predictions').glob('inat_*'), 'inat_'),
        ('AutoDDG', (TECH_ROOT / 'inaturalist').glob('autoddg_*'), 'autoddg_inat_'),
        ('DataAtlas', (TECH_ROOT / 'inaturalist').glob('dataatlas_*'), 'dataatlas_inat_'),
        ('Single LLM', M3_ROOT.glob('single_llm_inaturalist_*'), 'single_llm_inaturalist_'),
        ('ReAct', M3_ROOT.glob('react_inaturalist_*'), 'react_inaturalist_'),
    ]
    frames = []
    for method_name, run_dirs, prefix in specs:
        frame = inat_year_table(run_dirs, prefix)
        frame.insert(0, 'Method', method_name)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def print_all_inat_year_latex(df):
    latex_df = df.copy()
    latex_df['Method'] = latex_df['Method'].replace({'M³': 'M3PLACEHOLDER'})
    grouped = latex_df.set_index(['Method', 'Model'])
    latex = grouped.to_latex(
        index=True,
        escape=True,
        multirow=True,
        float_format='%.1f',
        column_format='llccc',
        caption=r'iNaturalist accuracy with coarser exact matches. A start or end year matches the year of the earliest or latest observation. A coordinate matches after both values are rounded to the nearest integer, with a fractional part of 0.5 or more rounding away from zero. The last column is the original exact coordinate match.',
        label='tab:all_models_inat_year',
    )
    old_header = (
        ' &  & Temp. year accuracy [\\%] & Spat. integer accuracy [\\%] & Spat. accuracy [\\%] \\\\\n'
        'Method & Model &  &  &  \\\\\n'
    )
    new_header = (
        'Method & Model & Temp. year accuracy [\\%] & Spat. integer accuracy [\\%] & Spat. accuracy [\\%] \\\\\n'
    )
    latex = latex.replace(old_header, new_header)
    latex = latex.replace(r'\cline{1-5}', r'\midrule')
    latex = latex.replace('\\midrule\n\\bottomrule', '\\bottomrule')
    latex = latex.replace('M3PLACEHOLDER', r'M$^3$')
    tex_path = TECH_ROOT / 'df_results_all_methods_inat_year.tex'
    tex_path.write_text(latex, encoding='utf-8')
    print(latex)
    print(f'Saved {tex_path}')
    return latex


all_inat_year = all_inat_year_methods()
print_all_inat_year_latex(all_inat_year)
all_inat_year.set_index(['Method', 'Model'])

\begin{table}
\caption{iNaturalist accuracy with coarser exact matches. A start or end year matches the year of the earliest or latest observation. A coordinate matches after both values are rounded to the nearest integer, with a fractional part of 0.5 or more rounding away from zero. The last column is the original exact coordinate match.}
\label{tab:all_models_inat_year}
\begin{tabular}{llccc}
\toprule
Method & Model & Temp. year accuracy [\%] & Spat. integer accuracy [\%] & Spat. accuracy [\%] \\
\midrule
\multirow[t]{4}{*}{M$^3$} & Llama3 70B & 81.0 & 99.8 & 99.0 \\
 & Mistral Devstral 2 24B & 69.5 & 100.0 & 99.2 \\
 & Qwen 3.5 122B & 76.0 & 100.0 & 99.2 \\
 & Qwen 3.6 35B & 87.0 & 96.0 & 95.2 \\
\midrule
\multirow[t]{5}{*}{AutoDDG} & Llama3 70B & 17.5 & 7.5 & 0.0 \\
 & Mistral Devstral 2 24B & 5.5 & 0.0 & 0.0 \\
 & Mistral Small 3.2 24B & 8.5 & 0.0 & 0.0 \\
 & Qwen 3.5 122B & 29.0 & 0.0 & 0.0 \\
 & Qwen 3.6 35B & 20.5 & 0.0 & 0.0 \\
\midrule
\multirow[t]{5}{*}{DataAtlas} & Llama3 

Temp. year accuracy [%]  \
Method     Model                                             
M³         Llama3 70B                                 81.0   
           Mistral Devstral 2 24B                     69.5   
           Qwen 3.5 122B                              76.0   
           Qwen 3.6 35B                               87.0   
AutoDDG    Llama3 70B                                 17.5   
           Mistral Devstral 2 24B                      5.5   
           Mistral Small 3.2 24B                       8.5   
           Qwen 3.5 122B                              29.0   
           Qwen 3.6 35B                               20.5   
DataAtlas  Llama3 70B                                 56.5   
           Mistral Devstral 2 24B                     64.5   
           Mistral Small 3.2 24B                      64.5   
           Qwen 3.5 122B                              59.0   
           Qwen 3.6 35B                               38.5   
Single LLM Llama3 70B                                  0.0   
           Mistral Devstral 2 24B                      0.0   
           Mistral Small 3.2 24B                       0.0   
           Qwen 3.5 122B                               0.0   
           Qwen 3.6 35B                                0.0   
ReAct      Llama3 70B                                 92.0   
           Mistral Devstral 2 24B                    100.0   
           Mistral Small 3.2 24B                      89.0   
           Qwen 3.5 122B                              97.0   
           Qwen 3.6 35B                               99.0   

                                   Spat. integer accuracy [%]  \
Method     Model                                                
M³         Llama3 70B                                    99.8   
           Mistral Devstral 2 24B                       100.0   
           Qwen 3.5 122B                                100.0   
           Qwen 3.6 35B                                  96.0   
AutoDDG    Llama3 70B                                     7.5   
           Mistral Devstral 2 24B                         0.0   
           Mistral Small 3.2 24B                          0.0   
           Qwen 3.5 122B                                  0.0   
           Qwen 3.6 35B                                   0.0   
DataAtlas  Llama3 70B                                     0.0   
           Mistral Devstral 2 24B                         0.0   
           Mistral Small 3.2 24B                          0.0   
           Qwen 3.5 122B                                  0.0   
           Qwen 3.6 35B                                   0.0   
Single LLM Llama3 70B                                     0.0   
           Mistral Devstral 2 24B                         0.0   
           Mistral Small 3.2 24B                          0.0   
           Qwen 3.5 122B                                  0.0   
           Qwen 3.6 35B                                   0.0   
ReAct      Llama3 70B                                   100.0   
           Mistral Devstral 2 24B                       100.0   
           Mistral Small 3.2 24B                         99.8   
           Qwen 3.5 122B                                 99.0   
           Qwen 3.6 35B                                  99.0   

                                   Spat. accuracy [%]  
Method     Model                                       
M³         Llama3 70B                            99.0  
           Mistral Devstral 2 24B                99.2  
           Qwen 3.5 122B                         99.2  
           Qwen 3.6 35B                          95.2  
AutoDDG    Llama3 70B                             0.0  
           Mistral Devstral 2 24B                 0.0  
           Mistral Small 3.2 24B                  0.0  
           Qwen 3.5 122B                          0.0  
           Qwen 3.6 35B                           0.0  
DataAtlas  Llama3 70B                             0.0  
           Mistral Devstral 2 24B                 0.0  
   